In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd


def build_week2_pipeline(
    input_file: str,
    output_file: str
) -> pd.DataFrame:
    """
    Read the Week 1 raw dataset, clean it, create features,
    and save the Week 2 processed dataset.
    """

    input_path = Path(input_file)
    output_path = Path(output_file)

    if not input_path.exists():
        raise FileNotFoundError(
            f"Input file was not found: {input_path.resolve()}"
        )

    df = pd.read_csv(input_path)

    # Standardize column names
    df.columns = [
        str(column).strip().replace(" ", "_")
        for column in df.columns
    ]

    # Parse date
    if "Date" in df.columns:
        df["Date"] = pd.to_datetime(
            df["Date"],
            errors="coerce"
        )
        df = df.sort_values("Date")

    # Convert relevant columns to numeric
    numeric_columns = [
        "Open",
        "High",
        "Low",
        "Close",
        "Adj_Close",
        "Volume",
        "VIX",
        "Treasury_10Y",
    ]

    for column in numeric_columns:
        if column in df.columns:
            df[column] = pd.to_numeric(
                df[column],
                errors="coerce"
            )

    # Remove duplicate dates
    if "Date" in df.columns:
        df = df.drop_duplicates(
            subset=["Date"],
            keep="last"
        )

    # Missing-value handling
    df = df.ffill()

    # Return features
    df["Daily_Return"] = df["Close"].pct_change()

    df["Log_Return"] = np.log(
        df["Close"] / df["Close"].shift(1)
    )

    # Annualized 20-day rolling volatility
    df["Rolling_Volatility_20D"] = (
        df["Log_Return"]
        .rolling(window=20)
        .std()
        * np.sqrt(252)
    )

    # VIX features
    if "VIX" in df.columns:
        df["VIX_Change"] = df["VIX"].pct_change()

        df["VIX_JPM_Correlation_20D"] = (
            df["Daily_Return"]
            .rolling(window=20)
            .corr(df["VIX_Change"])
        )

    # Interest-rate feature
    if "Treasury_10Y" in df.columns:
        df["Rate_Change"] = (
            df["Treasury_10Y"].diff()
        )

    # Volume feature
    if "Volume" in df.columns:
        df["Volume_Change"] = (
            df["Volume"].pct_change()
        )

    # Replace infinities
    df = df.replace(
        [np.inf, -np.inf],
        np.nan
    )

    # Remove rows missing required calculated features
    required_features = [
        "Daily_Return",
        "Log_Return",
        "Rolling_Volatility_20D",
    ]

    df = df.dropna(
        subset=[
            column
            for column in required_features
            if column in df.columns
        ]
    )

    # Create output directory automatically
    output_path.parent.mkdir(
        parents=True,
        exist_ok=True
    )

    df.to_csv(
        output_path,
        index=False
    )

    print(f"Input rows: {len(pd.read_csv(input_path))}")
    print(f"Output rows: {len(df)}")
    print(f"Saved to: {output_path.resolve()}")

    return df